# Black-Box Optimisation Capstone: Reproducible Analysis

**Imperial College London Professional Certificate in Machine Learning and Artificial Intelligence**

This notebook provides a reproducible analysis of the recorded Stage 2 Black-Box Optimisation (BBO) capstone history.

It has two purposes:

1. reproduce the analysis of the initial data and the 13 submitted query rounds; and
2. demonstrate the Gaussian Process / Bayesian optimisation workflow used to support query selection.

> **Reproducibility note:** the hidden objective functions were evaluated externally through the capstone portal and are therefore not available in this repository. The notebook reproduces the analysis and surrogate-modelling workflow from the recorded observations; it does not recreate the hidden functions themselves.

The notebook deliberately distinguishes the **recorded historical submissions** from the **representative optimisation code** used to explain and reproduce the methodology.

## 1. Imports and configuration

The project uses a lightweight scientific Python stack: NumPy and Pandas for data handling, Matplotlib for visualisation, SciPy for Sobol sampling and probability functions, and scikit-learn for Gaussian Process regression.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm, qmc
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

RANDOM_STATE = 42
np.set_printoptions(precision=6, suppress=True)

# The notebook is designed to run from the notebooks/ directory.
REPO_ROOT = Path(".")
DATA_DIR = REPO_ROOT / "data"
INITIAL_DIR = DATA_DIR / "initial"

print(f"Repository root: {REPO_ROOT.resolve()}")

## 2. Load the recorded query history

`queries.csv` contains the 13 Stage 2 query rounds (Modules 12–24), while `results.csv` contains the corresponding portal outputs.

The input dimensions differ by function, so the query table uses columns `x1` through `x8` and leaves unused dimensions blank.

In [ ]:
queries = pd.read_csv(DATA_DIR / "queries.csv")
results = pd.read_csv(DATA_DIR / "results.csv")

history = queries.merge(
    results,
    on=["round", "module", "function"],
    how="inner"
)

history.head(12)

In [ ]:
print(f"Recorded query rounds: {history['round'].nunique()}")
print(f"Functions: {history['function'].nunique()}")
print(f"Recorded evaluations: {len(history)}")

history.groupby("function")["output"].agg(["count", "min", "max"])

## 3. Load the initial capstone data

The original `.npy` files supplied for the challenge are retained in `data/initial/`.

These initial observations are combined with the weekly submitted queries when fitting the final retrospective surrogate models.

In [ ]:
DIMENSIONS = {
    "F1": 2,
    "F2": 2,
    "F3": 3,
    "F4": 4,
    "F5": 4,
    "F6": 5,
    "F7": 6,
    "F8": 8,
}

initial_data = {}

for i in range(1, 9):
    function = f"F{i}"
    X = np.load(INITIAL_DIR / f"function{i}_initial_inputs.npy")
    y = np.load(INITIAL_DIR / f"function{i}_initial_outputs.npy")
    initial_data[function] = (X, y)
    print(
        f"{function}: X shape={X.shape}, y shape={y.shape}, "
        f"initial best={np.max(y):.6g}"
    )

## 4. Combine initial observations with Stage 2 queries

For each function, the recorded weekly queries are appended to its original capstone data.

This produces the final dataset available at the end of the optimisation challenge.

In [ ]:
def weekly_xy(function):
    d = DIMENSIONS[function]
    part = history.loc[history["function"] == function].sort_values("round")
    X = part[[f"x{i}" for i in range(1, d + 1)]].to_numpy(dtype=float)
    y = part["output"].to_numpy(dtype=float)
    return X, y


def complete_xy(function):
    X0, y0 = initial_data[function]
    Xw, yw = weekly_xy(function)
    return np.vstack([X0, Xw]), np.concatenate([y0, yw])


for function in DIMENSIONS:
    X, y = complete_xy(function)
    print(f"{function}: final dataset shape = {X.shape}, best observed = {np.max(y):.6g}")

## 5. Evolution of the submitted outputs

The plots below show the outputs generated by the 13 Stage 2 submissions.

Because the objective scales differ substantially between functions, each function is plotted separately.

In [ ]:
for function in DIMENSIONS:
    part = history.loc[history["function"] == function].sort_values("round")
    plt.figure(figsize=(8, 4))
    plt.plot(part["round"], part["output"], marker="o")
    plt.xlabel("Query round")
    plt.ylabel("Objective value")
    plt.title(f"{function}: Stage 2 output history")
    plt.grid(alpha=0.25)
    plt.show()

### Observed patterns

The final history demonstrates several distinct optimisation behaviours:

- **F5** benefited strongly from continued local refinement and finished at its highest submitted value.
- **F2** was sensitive to small coordinate changes and recovered after returning to a previously stronger neighbourhood.
- **F7** deteriorated during repeated movement in one direction, then recovered strongly in the final round after backtracking.
- **F8** entered a very flat region around 9.57, consistent with diminishing returns.
- Higher-dimensional functions generally remained harder to interpret visually and required greater reliance on modelling and historical evidence.

In [ ]:
summary = (
    history.sort_values(["function", "round"])
    .groupby("function")
    .agg(
        first_stage2_output=("output", "first"),
        final_output=("output", "last"),
        best_stage2_output=("output", "max"),
    )
)

best_rounds = (
    history.loc[history.groupby("function")["output"].idxmax(), ["function", "round"]]
    .set_index("function")
    .rename(columns={"round": "best_round"})
)

summary = summary.join(best_rounds)
summary["change_first_to_final"] = (
    summary["final_output"] - summary["first_stage2_output"]
)
summary

## 6. Gaussian Process surrogate

The core surrogate is a Gaussian Process with a Matérn kernel.

A small WhiteKernel term is included in this **retrospective reproducible implementation** to allow for local irregularity/noise. Exact historical kernel settings were not identical for every function and round, so this notebook presents a consistent final implementation rather than pretending that one fixed configuration was used unchanged throughout all 13 weeks.

In [ ]:
def build_gp(n_dimensions, random_state=RANDOM_STATE):
    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=np.ones(n_dimensions),
            length_scale_bounds=(1e-2, 1e2),
            nu=1.5,
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e0),
        )
    )

    return GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=5,
        random_state=random_state,
    )

## 7. Acquisition functions

Two acquisition ideas used during the project are implemented below.

### Expected Improvement (EI)

EI balances the amount by which a candidate may improve the current best result with the model's uncertainty.

### Upper Confidence Bound (UCB)

UCB combines the predicted mean with a multiple of predictive uncertainty.

The parameters `xi` and `beta` allow the exploration–exploitation balance to be adjusted.

In [ ]:
def expected_improvement(mu, sigma, best_observed, xi=0.01):
    improvement = mu - best_observed - xi

    with np.errstate(divide="ignore", invalid="ignore"):
        z = np.divide(
            improvement,
            sigma,
            out=np.zeros_like(improvement),
            where=sigma > 0,
        )

    ei = improvement * norm.cdf(z) + sigma * norm.pdf(z)
    ei[sigma <= 0] = 0.0
    return ei


def upper_confidence_bound(mu, sigma, beta=2.5):
    return mu + beta * sigma

## 8. Sobol candidate generation

The hidden functions operate on inputs constrained to `[0, 1]`.

Sobol low-discrepancy sequences provide structured multidimensional candidate coverage without evaluating the hidden function itself.

In [ ]:
def sobol_candidates(n_dimensions, power=14, seed=RANDOM_STATE):
    sampler = qmc.Sobol(
        d=n_dimensions,
        scramble=True,
        seed=seed,
    )
    return sampler.random_base2(m=power)


example_candidates = sobol_candidates(2, power=10)
example_candidates.shape

## 9. Representative query recommender

The function below demonstrates the reproducible query-selection workflow:

1. combine the available observations;
2. fit a GP;
3. generate Sobol candidates;
4. predict mean and uncertainty;
5. score candidates using EI or UCB;
6. return the strongest candidate.

This is a **representative implementation of the methodology**, not a claim that the exact same code and hyperparameters generated every historical weekly submission.

In [ ]:
def recommend_query(
    function,
    acquisition="ei",
    xi=0.01,
    beta=2.5,
    candidate_power=14,
    seed=RANDOM_STATE,
):
    X, y = complete_xy(function)
    d = DIMENSIONS[function]

    gp = build_gp(d)
    gp.fit(X, y)

    candidates = sobol_candidates(d, power=candidate_power, seed=seed)
    mu, sigma = gp.predict(candidates, return_std=True)

    if acquisition.lower() == "ei":
        score = expected_improvement(
            mu,
            sigma,
            best_observed=np.max(y),
            xi=xi,
        )
    elif acquisition.lower() == "ucb":
        score = upper_confidence_bound(mu, sigma, beta=beta)
    else:
        raise ValueError("acquisition must be 'ei' or 'ucb'")

    idx = int(np.argmax(score))

    return {
        "function": function,
        "candidate": candidates[idx],
        "predicted_mean": float(mu[idx]),
        "predicted_std": float(sigma[idx]),
        "acquisition_score": float(score[idx]),
        "fitted_kernel": str(gp.kernel_),
    }

## 10. Example: retrospective Function 5 recommendation

Function 5 showed the clearest sustained improvement during the challenge, progressing from 892.91 in the first Stage 2 query to 1368.74 in the final round.

The cell below demonstrates what the final retrospective GP + EI model recommends after seeing the complete dataset.

In [ ]:
f5_recommendation = recommend_query(
    "F5",
    acquisition="ei",
    xi=0.01,
)

f5_recommendation

The recommendation above should **not** be interpreted as another official capstone submission. The optimisation challenge has finished. It demonstrates how the final accumulated data can be passed through the documented surrogate and acquisition workflow.

## 11. 2D surrogate visualisation

Functions 1 and 2 can be visualised directly.

The helper below plots the GP posterior mean across the two-dimensional input space together with the observed points.

This is particularly useful for showing why lower-dimensional optimisation is easier to interpret than the 5D–8D functions.

In [ ]:
def plot_2d_surrogate(function, grid_size=80):
    if DIMENSIONS[function] != 2:
        raise ValueError("This visualisation is only available for 2D functions.")

    X, y = complete_xy(function)
    gp = build_gp(2)
    gp.fit(X, y)

    axis = np.linspace(0, 1, grid_size)
    xx, yy = np.meshgrid(axis, axis)
    grid = np.column_stack([xx.ravel(), yy.ravel()])

    mu = gp.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(7, 6))
    contour = plt.contourf(xx, yy, mu, levels=30)
    plt.colorbar(contour, label="GP predicted mean")
    plt.scatter(X[:, 0], X[:, 1], s=35, edgecolors="black")
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(f"{function}: retrospective GP surrogate")
    plt.show()


plot_2d_surrogate("F2")

## 12. Final Stage 2 outputs

The final official query round produced the following outputs.

In [ ]:
final_results = (
    history.loc[history["round"] == history["round"].max(), ["function", "output"]]
    .set_index("function")
)

final_results

### Final-round interpretation

The final round provided several useful confirmations:

- **F5:** continued exploitation produced a new Stage 2 best of approximately **1368.74**.
- **F2:** returning to the previously successful neighbourhood produced a recovery to approximately **0.5431**.
- **F7:** backtracking towards an earlier promising region increased the result to approximately **1.3466**, reversing the late-round decline.
- **F8:** returned to **9.570335**, reinforcing the interpretation of a late-stage plateau.
- **F6:** the final local move did not improve the result, illustrating that exploitation does not guarantee monotonic improvement.

## 13. Limitations

The analysis has several important limitations:

- The hidden objective functions are unavailable, so true global optima cannot be verified locally.
- The datasets are small relative to the dimensionality of several functions.
- Later queries are intentionally concentrated around promising regions, creating sequential sampling bias.
- GP recommendations depend on kernel and noise assumptions.
- The retrospective implementation standardises parts of the GP configuration for reproducibility and should not be interpreted as an exact archival copy of every weekly code revision.
- Strong observed values demonstrate successful search behaviour, but do not prove that a global maximum was found.

## 14. Key lessons

The capstone reinforced several practical optimisation principles:

1. **Exploration is most valuable when enough future evaluations remain to use the information gained.**
2. **Uncertainty should be treated as information rather than ignored.**
3. **Historical evidence should remain available even when recent queries move elsewhere.**
4. **Different functions can require different exploration policies.**
5. **Increasing model complexity is only useful when the amount and quality of data justify it.**
6. **Optimisation is a sequential decision-making process, not simply a search for the largest immediate prediction.**

The final methodology therefore combined Bayesian optimisation with practical judgement, using the surrogate model to inform decisions rather than treating it as an unquestionable optimiser.

## 15. Repository documentation

Additional context is available in:

- `README.md` — project overview and non-technical summary
- `DATASHEET.md` — data provenance, composition and limitations
- `MODEL_CARD.md` — optimisation approach, intended use and constraints
- `METHODOLOGY.md` — detailed technical methodology
- `OPTIMISATION_HISTORY.md` — evolution of the strategy and outcomes across the project

Together, the notebook and documentation provide a transparent record of both the **technical implementation** and the **decision-making process** behind the BBO capstone.